# 🚀 Banglish Emotion Detection: Phase 4

**Transformers & The Dual-Input Architecture**

In [ ]:
# 1. Setup Environment
import sys
import torch
import pandas as pd
import numpy as np
from google.colab import drive

!pip install transformers datasets accelerate -q

# Mount Drive
drive.mount('/content/drive')
sys.path.append('/content/drive/My Drive/424 Project')

from helpers.architecture import DualStreamEmotionModel, FocalLoss

In [ ]:
# 2. Load the Transliteration Model (Idea 1)
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("Loading Banglish-to-Bengali Transliterator...")
translit_tokenizer = AutoTokenizer.from_pretrained("Mdkaif2782/banglish-to-bangla")
translit_model = AutoModelForSeq2SeqLM.from_pretrained("Mdkaif2782/banglish-to-bangla")
translit_model.to("cuda" if torch.cuda.is_available() else "cpu")

def transliterate_banglish(texts, batch_size=32):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    results = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = translit_tokenizer(batch, return_tensors="pt", padding=True, truncation=True).to(device)
        outputs = translit_model.generate(**inputs, max_length=128)
        results.extend(translit_tokenizer.batch_decode(outputs, skip_special_tokens=True))
    return results

In [ ]:
# 3. Example Transliteration
sample_texts = ["ami bhalo achi", "amar onek kosto hocche"]
transliterated = transliterate_banglish(sample_texts)
print("Banglish:", sample_texts)
print("Bengali:", transliterated)

In [ ]:
# 4. Prepare Data Loaders
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

class DualInputDataset(Dataset):
    def __init__(self, banglish_texts, bengali_texts, lexicon_features, labels, 
                 xlmr_tokenizer, banglabert_tokenizer, max_len=128):
        self.banglish_texts = banglish_texts
        self.bengali_texts = bengali_texts
        self.lexicon_features = lexicon_features
        self.labels = labels
        self.xlmr_tokenizer = xlmr_tokenizer
        self.banglabert_tokenizer = banglabert_tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        banglish_text = str(self.banglish_texts[idx])
        bengali_text = str(self.bengali_texts[idx])
        lexicon = torch.tensor(self.lexicon_features[idx], dtype=torch.float)
        label = torch.tensor(self.labels[idx], dtype=torch.long)

        # Stream A (XLM-R on Banglish)
        encoding_a = self.xlmr_tokenizer.encode_plus(
            banglish_text, add_special_tokens=True, max_length=self.max_len, 
            padding='max_length', return_attention_mask=True, truncation=True, return_tensors='pt'
        )
        
        # Stream B (BanglaBERT on Bengali)
        encoding_b = self.banglabert_tokenizer.encode_plus(
            bengali_text, add_special_tokens=True, max_length=self.max_len,
            padding='max_length', return_attention_mask=True, truncation=True, return_tensors='pt'
        )

        return {
            'input_ids_a': encoding_a['input_ids'].flatten(),
            'attention_mask_a': encoding_a['attention_mask'].flatten(),
            'input_ids_b': encoding_b['input_ids'].flatten(),
            'attention_mask_b': encoding_b['attention_mask'].flatten(),
            'lexicon': lexicon,
            'label': label
        }

### ⚠️ NOTE TO RESEARCHER:
To proceed, you need to transliterate the *entire* Banglish dataset using `transliterate_banglish()`. 
Since it's 80K samples, this will take some time. We suggest doing this offline or in a dedicated Colab session and saving it.